# VoxTell on Neurite-OASIS — Local Execution (Apple Silicon M5 MacBook Pro)

Runs the complete **Axis II: Sensitivity to Prompt Variation & Uncertainty Estimation** suite locally on your M5 MacBook Pro (using Apple Silicon `mps` GPU acceleration or `cpu`), while keeping the Kaggle notebook (`voxtell-oasis-mechanistic-experiments-kaggle.ipynb`) untouched.

- **Local Repo Root:** `/Users/lifeasmeher/Project/positional_bias/positional-bias-medical-vlms`
- **Local Dataset:** `/Users/lifeasmeher/Project/positional_bias/positional-bias-medical-vlms/data/neurite_oasis_100` (auto-downloads from Hugging Face via `.env` `HF_TOKEN` if not already present)
- **Local VoxTell Source:** `/Users/lifeasmeher/Project/positional_bias/VoxTell-main`

## 1. Install / Verify Local Dependencies (Without Downgrading `torch`)

`VoxTell-main` pins `torch<2.9` in its `pyproject.toml`, whereas your local `.venv` runs `torch 2.14.0` with native M5 MPS 3D convolution support. The cell below installs `VoxTell-main` with `--no-deps` and installs only the lightweight runtime dependencies (`einops`, `positional-encodings`, `dynamic-network-architectures`, `acvl-utils`, `batchgenerators`, `nnunetv2`, `transformers<5`, `scipy`) so your local `torch 2.14.0` stays intact.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('/Users/lifeasmeher/Project/positional_bias/positional-bias-medical-vlms')
VOXTELL_LOCAL_SRC = Path('/Users/lifeasmeher/Project/positional_bias/VoxTell-main')

def _ensure_local_packages():
    missing = []
    for mod, pkg in [
        ('einops', 'einops'),
        ('positional_encodings', 'positional-encodings[pytorch]'),
        ('dynamic_network_architectures', 'dynamic-network-architectures>=0.4.2,<0.5.0'),
        ('acvl_utils', 'acvl-utils>=0.2.3,<0.3.0'),
        ('batchgenerators', 'batchgenerators'),
        ('nnunetv2', 'nnunetv2'),
        ('transformers', 'transformers>=4.54.0,<5'),
        ('huggingface_hub', 'huggingface_hub'),
        ('scipy', 'scipy'),
    ]:
        try:
            __import__(mod)
        except ImportError:
            missing.append(pkg)

    if missing:
        print('Installing missing runtime packages:', missing)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

    try:
        import voxtell
    except ImportError:
        if VOXTELL_LOCAL_SRC.exists():
            print('Installing local VoxTell-main in editable mode (--no-deps to preserve torch 2.14)...')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(VOXTELL_LOCAL_SRC)])
        else:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'voxtell'])

_ensure_local_packages()

import torch
import voxtell
print('Python executable:', sys.executable)
print('PyTorch version:', torch.__version__)
print('Apple Silicon MPS available:', torch.backends.mps.is_available())
print('VoxTell loaded from:', voxtell.__file__)


## 2. Load `.env` & Configure Local Directories

Reads `HF_TOKEN` from `/Users/lifeasmeher/Project/positional_bias/positional-bias-medical-vlms/.env` and sets up:
- `RESEARCH_REPO`: `/Users/lifeasmeher/Project/positional_bias/positional-bias-medical-vlms`
- `NEURITE_OASIS_DIR`: `/Users/lifeasmeher/Project/positional_bias/positional-bias-medical-vlms/data/neurite_oasis_100`
- `RESULTS_ROOT`: `/Users/lifeasmeher/Project/positional_bias/positional-bias-medical-vlms/results_axis2`

In [ ]:
import os
import sys
from pathlib import Path

RESEARCH_REPO = Path('/Users/lifeasmeher/Project/positional_bias/positional-bias-medical-vlms')
SCRIPTS_DIR = RESEARCH_REPO / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# Load .env variables (HF_TOKEN, KAGGLE_API_TOKEN, GITHUB_PAT)
ENV_FILE = RESEARCH_REPO / '.env'
if ENV_FILE.exists():
    for line in ENV_FILE.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

NEURITE_OASIS_DIR = RESEARCH_REPO / 'data' / 'neurite_oasis_100'
RESULTS_ROOT = RESEARCH_REPO / 'results_axis2'
CKPT_CACHE_DIR = RESEARCH_REPO / 'data' / 'checkpoints'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('RESEARCH_REPO:', RESEARCH_REPO)
print('NEURITE_OASIS_DIR:', NEURITE_OASIS_DIR, '(exists:', NEURITE_OASIS_DIR.exists(), ')')
print('RESULTS_ROOT:', RESULTS_ROOT)


## 3. Verify / Download 100-Subject Neurite-OASIS Dataset

If `data/neurite_oasis_100` is already downloaded locally (200 `.nii.gz` volume files + `manifest_neurite_oasis_100.csv`), this cell verifies it immediately without re-downloading. Otherwise, it pulls `Fatema142/neurite-oasis-100-rq-analysis` from Hugging Face.

In [ ]:
from huggingface_hub import snapshot_download

_vol_dir = NEURITE_OASIS_DIR / 'volumes'
if _vol_dir.exists() and len(list(_vol_dir.glob('*.nii.gz'))) >= 200:
    print('Local Neurite-OASIS dataset already present at:', NEURITE_OASIS_DIR)
else:
    _hf_token = os.environ.get('HF_TOKEN')
    assert _hf_token, 'Set HF_TOKEN in .env before downloading Fatema142/neurite-oasis-100-rq-analysis'
    NEURITE_OASIS_DIR = Path(snapshot_download(
        repo_id='Fatema142/neurite-oasis-100-rq-analysis',
        repo_type='dataset',
        token=_hf_token,
        local_dir=str(NEURITE_OASIS_DIR),
    ))

print('Total volume files:', len(list((NEURITE_OASIS_DIR / 'volumes').glob('*.nii.gz'))))


## 4. Download & Load VoxTell (`voxtell_v1.1`) on M5 Apple Silicon (`mps` / `cpu`)

Downloads `mrokuss/VoxTell` (`voxtell_v1.1`) into `data/checkpoints/voxtell_v1.1` (cached locally across runs) and loads `VoxTellPredictor` using `load_voxtell_predictor_safe` from `scripts/run_axis2_oasis.py`.

In [ ]:
import torch
from huggingface_hub import snapshot_download
from run_axis2_oasis import load_voxtell_predictor_safe

VOXTELL_MODEL_VERSION = 'voxtell_v1.1'
_local_repo_dir = snapshot_download(
    repo_id='mrokuss/VoxTell',
    allow_patterns=[f'{VOXTELL_MODEL_VERSION}/*'],
    local_dir=str(CKPT_CACHE_DIR),
)
VOXTELL_MODEL_DIR = Path(_local_repo_dir) / VOXTELL_MODEL_VERSION
print('VoxTell checkpoint at:', VOXTELL_MODEL_DIR)

# Automatically selects 'mps' on M5 MacBook Pro (or override with DEVICE = 'cpu')
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
predictor = load_voxtell_predictor_safe(
    model_dir=VOXTELL_MODEL_DIR,
    device=DEVICE,
    fp16_text_backbone=True,
)
print('VoxTell predictor ready on device:', predictor.device)


## 5. Deliverable 1 (`exp0_frozen_prompts`): Verify Git Synonym Manifest & Export Frozen Prompt Table

Loads `manifests/manifest_neurite_oasis_100_synonyms.csv` (19,000 rows across all 100 subjects) and exports the frozen prompt tables (`CSV`, `JSON`, `MD`) into `results_axis2/exp0_frozen_prompts/`.

In [ ]:
import pandas as pd
from axis2_prompts import export_frozen_prompt_table

GIT_SYNONYM_MANIFEST = RESEARCH_REPO / 'manifests' / 'manifest_neurite_oasis_100_synonyms.csv'
syn_df = pd.read_csv(GIT_SYNONYM_MANIFEST)
print('Loaded Git Synonym Manifest:', GIT_SYNONYM_MANIFEST)
print('Shape:', syn_df.shape, '| Unique subjects:', syn_df['subject_id'].nunique(), '| Unique prompts:', syn_df['prompt_id'].nunique())

exp0_paths = export_frozen_prompt_table(
    output_dir=RESULTS_ROOT / 'exp0_frozen_prompts',
    oasis_manifest_csv=NEURITE_OASIS_DIR / 'manifest_neurite_oasis_100.csv',
)
for k, v in exp0_paths.items():
    print(f'  {k}: {v}')


## 6. Local Experiment Configuration

Because every experiment saves per-subject checkpoints in `results_axis2/<experiment>/checkpoints/<subject_id>.json`, you can start with a small local batch (e.g. `MAX_SUBJECTS = 5`) to inspect outputs quickly, and then increase `MAX_SUBJECTS = 100` anytime — already-completed subjects are automatically resumed without recomputing!

In [ ]:
# 'core_7' (recommended), 'extended_11', or 'all_16'
STRUCTURE_SCOPE = 'core_7'

# Set MAX_SUBJECTS = 5 for a fast local run, or MAX_SUBJECTS = 100 for all 100 subjects.
# Completed subjects are cached in checkpoints/ and skipped automatically when RESUME = True.
MAX_SUBJECTS = 5
START_INDEX = 0
END_INDEX = None

# Number of stochastic forward passes for Experiment IV (MC Dropout Uncertainty)
MC_DROPOUT_PASSES = 10
SPATIAL_DROPOUT_P = 0.0

# Number of subjects to save full 3D .nii.gz files for (all subjects save lossless RLE + .npz)
SAVE_NIFTI_SUBJECTS = 3
RESUME = True


## 7. Experiment I (`exp1_side_word_comparison`): Does the Side Word Change the Output?

Runs Base (no-side), Left, Right, Contradictory (`left X on right hemisphere`), and Absent-Organ (`kidney`, `adrenal tumor`, `lung nodule`) prompts.
- Output folder: `results_axis2/exp1_side_word_comparison/` (`metrics.csv`, `summary.csv`, `results.json`, `summary.json`, `REPORT.md`, `masks/`).

In [ ]:
from axis2_experiments import run_experiment_1_side_words

exp1_result = run_experiment_1_side_words(
    predictor=predictor,
    neurite_oasis_dir=NEURITE_OASIS_DIR,
    results_root=RESULTS_ROOT,
    structure_scope=STRUCTURE_SCOPE,
    max_subjects=MAX_SUBJECTS,
    start_index=START_INDEX,
    end_index=END_INDEX,
    save_nifti_subjects=SAVE_NIFTI_SUBJECTS,
    save_rle_in_csv=True,
    resume=RESUME,
)
pd.DataFrame(exp1_result['summary']).head(20)


## 8. Experiments II & III (`exp2_synonym_shift` & `exp3_layer_localization`): Synonym Shift Analysis + 14-Stage Layer Localization

Runs jointly in a single encoder pass per volume:
- **Experiment II (`results_axis2/exp2_synonym_shift/`):** Embedding shift at Frozen Qwen3 & Prompt Decoder vs. Prediction shift (`1 - Dice`, volume shift, centroid shift, similarity map correlation, and Pearson/Spearman correlations).
- **Experiment III (`results_axis2/exp3_layer_localization/`):** 14-stage feature distance curve across Qwen3, `project_text_embed`, Transformer Decoder layers 1–6 (plus cross-attention divergence), `LayerNorm`, and U-Net fusion stages 0–4.

In [ ]:
from axis2_experiments import run_experiment_2_and_3_synonyms_and_layers

exp23_result = run_experiment_2_and_3_synonyms_and_layers(
    predictor=predictor,
    neurite_oasis_dir=NEURITE_OASIS_DIR,
    results_root=RESULTS_ROOT,
    structure_scope=STRUCTURE_SCOPE,
    max_subjects=MAX_SUBJECTS,
    start_index=START_INDEX,
    end_index=END_INDEX,
    save_nifti_subjects=SAVE_NIFTI_SUBJECTS,
    save_rle_in_csv=True,
    resume=RESUME,
)
print('Experiment II Correlations (Embedding Distance vs. Prediction Shift):')
display(pd.DataFrame(exp23_result['exp2']['correlations']))
print('\nExperiment III Layer Localization Summary (Left vs Right):')
df_exp3 = pd.DataFrame(exp23_result['exp3']['summary'])
display(df_exp3[df_exp3['comparison_type'] == 'left_vs_right'])


## 9. Experiment IV (`exp4_mc_dropout_uncertainty`): MC Dropout Uncertainty Estimation

Unfreezes VoxTell's dropout layers and runs `MC_DROPOUT_PASSES` stochastic forward passes per prompt.
- Computes **Epistemic Uncertainty** (predictive variance & mutual information) and **Aleatoric Uncertainty** (average per-pass binary entropy), inter-pass Dice stability, and ECE.
- Output folder: `results_axis2/exp4_mc_dropout_uncertainty/`.

In [ ]:
from axis2_experiments import run_experiment_4_mc_dropout_uncertainty

exp4_result = run_experiment_4_mc_dropout_uncertainty(
    predictor=predictor,
    neurite_oasis_dir=NEURITE_OASIS_DIR,
    results_root=RESULTS_ROOT,
    structure_scope=STRUCTURE_SCOPE,
    n_passes=MC_DROPOUT_PASSES,
    spatial_dropout_p=SPATIAL_DROPOUT_P,
    max_subjects=MAX_SUBJECTS,
    start_index=START_INDEX,
    end_index=END_INDEX,
    save_nifti_subjects=SAVE_NIFTI_SUBJECTS,
    save_rle_in_csv=True,
    resume=RESUME,
)
print('Wording Stability & Contradictory Prompt Uncertainty:')
display(pd.DataFrame(exp4_result['stability']))


## 10. Experiment V (`exp5_decoder_ablation`): 4-Way Image vs. Text Decoder Ablation

Evaluates `both_on`, `image_only`, `text_only`, and `both_off` across Experiments I & II.
- Output folder: `results_axis2/exp5_decoder_ablation/`.

In [ ]:
from axis2_experiments import run_experiment_5_decoder_ablation

exp5_result = run_experiment_5_decoder_ablation(
    predictor=predictor,
    neurite_oasis_dir=NEURITE_OASIS_DIR,
    results_root=RESULTS_ROOT,
    structure_scope=STRUCTURE_SCOPE,
    max_subjects=MAX_SUBJECTS,
    start_index=START_INDEX,
    end_index=END_INDEX,
    save_nifti_subjects=SAVE_NIFTI_SUBJECTS,
    save_rle_in_csv=True,
    resume=RESUME,
)
display(pd.DataFrame(exp5_result['summary']))


## 11. Inspect All Local Output Folders (`results_axis2/`)

In [ ]:
for sub in sorted(RESULTS_ROOT.iterdir()):
    if sub.is_dir():
        files = [p.name for p in sorted(sub.iterdir())]
        print(f'{sub.name}/ -> {files}')
